# VoiceGrad 论文同口径 Closed-set CER 评测（Colab）

这个 notebook 按论文的 **closed-set** 设定来做两件事：

1. 用指定 checkpoint（默认 `model_epoch_1000.pt`）做 **12 个有方向的 cross-speaker 转换**
   - 闭集说话人：`clb, bdl, slt, rms`
   - 每个 source 转到另外 3 个 target
   - 每个说话人使用 **32 条 evaluation utterances**
   - 总数应为 **4 × 32 × 3 = 384 条 wav**

2. 用 **wav2vec 2.0 Large LV-60K**（`facebook/wav2vec2-large-960h-lv60-self`）来做 ASR 转写，再计算 CER

'epochs': 4000,'batch_size': 16,'lr': 1e-4


In [ ]:
# ===== Cell 1: 挂载 Google Drive =====
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ===== Cell 2: 安装依赖 =====
!pip -q install transformers jiwer soundfile librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 94.8 MB/s eta 0:00:00


In [ ]:
# ===== Cell 3: 路径配置（先改这里） =====
import os
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/VoiceGrad"
DATA_ROOT = os.path.join(PROJECT_ROOT, "data")

# 选择要评测的 checkpoint
CKPT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
CKPT_PATH = os.path.join(CKPT_DIR, "model_epoch_1000.pt")

# HiFi-GAN
HIFIGAN_ROOT = os.path.join(PROJECT_ROOT, "hifi-gan")
HIFIGAN_CONFIG = os.path.join(HIFIGAN_ROOT, "config_16k.json")
HIFIGAN_CKPT = os.path.join(HIFIGAN_ROOT, "g_00105000")   # 改成你自己的真实权重

# 参考文本（建议直接填 CMU ARCTIC 的 txt.done.data）
TEXT_PATH = "/content/drive/MyDrive/VoiceGrad/data/cmuarctic.data.txt"
# 例如：
# TEXT_PATH = "/content/drive/MyDrive/VoiceGrad/cmu_us_arctic/etc/txt.done.data"

# 输出目录
RUN_NAME = "ckpt1000_closedset_cer_paper_protocol"
OUT_ROOT = os.path.join(PROJECT_ROOT, RUN_NAME)
GEN_WAV_ROOT = os.path.join(OUT_ROOT, "generated_wavs")
os.makedirs(GEN_WAV_ROOT, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)
print("CKPT_PATH    =", CKPT_PATH)
print("HIFIGAN_ROOT =", HIFIGAN_ROOT)
print("TEXT_PATH    =", TEXT_PATH)
print("OUT_ROOT     =", OUT_ROOT)
print()
print("PROJECT exists? ", os.path.isdir(PROJECT_ROOT))
print("DATA exists?    ", os.path.isdir(DATA_ROOT))
print("CKPT exists?    ", os.path.exists(CKPT_PATH))
print("HIFIGAN exists? ", os.path.isdir(HIFIGAN_ROOT))

PROJECT_ROOT = /content/drive/MyDrive/VoiceGrad
DATA_ROOT    = /content/drive/MyDrive/VoiceGrad/data
CKPT_PATH    = /content/drive/MyDrive/VoiceGrad/checkpoints/model_epoch_1000.pt
HIFIGAN_ROOT = /content/drive/MyDrive/VoiceGrad/hifi-gan
TEXT_PATH    = /content/drive/MyDrive/VoiceGrad/data/cmuarctic.data.txt
OUT_ROOT     = /content/drive/MyDrive/VoiceGrad/ckpt1000_closedset_cer_paper_protocol

PROJECT exists?  True
DATA exists?     True
CKPT exists?     True
HIFIGAN exists?  True


In [ ]:
# ===== Cell 4: 基础导入 =====
import os
import sys
import re
import json
import glob
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import soundfile as sf
import librosa
from pathlib import Path
from tqdm import tqdm

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("cwd =", os.getcwd())
print("DEVICE =", DEVICE)

cwd = /content/drive/MyDrive/VoiceGrad
DEVICE = cuda


In [ ]:
# ===== Cell 5: 导入当前项目代码（从你 Drive 里的最新代码读取） =====
from model import VoiceGrad
from diffusion import VoiceGradDiffusion

In [ ]:
# ===== Cell 6: 通用函数 =====
def ensure_mel_shape(mel):
    if mel.ndim != 2:
        raise ValueError(f"mel shape wrong: {mel.shape}")
    if mel.shape[0] == 80:
        return mel
    if mel.shape[1] == 80:
        return mel.T
    raise ValueError(f"mel shape wrong: {mel.shape}")

def ensure_bnf_shape(bnf):
    if bnf.ndim != 2:
        raise ValueError(f"bnf shape wrong: {bnf.shape}")
    if bnf.shape[0] == 144:
        return bnf
    if bnf.shape[1] == 144:
        return bnf.T
    raise ValueError(f"bnf shape wrong: {bnf.shape}")

def resample_bnf_to_mel_len(bnf, mel_len):
    # 和当前训练时的 dataset.py 一致
    bnf_tensor = torch.from_numpy(bnf).float().unsqueeze(0)
    bnf_tensor = F.interpolate(
        bnf_tensor,
        size=mel_len,
        mode="linear",
        align_corners=False
    )
    return bnf_tensor.squeeze(0).cpu().numpy()

def load_stats(data_root):
    mean = np.load(os.path.join(data_root, "stats", "mel_mean.npy"))
    std = np.load(os.path.join(data_root, "stats", "mel_std.npy"))
    mean = torch.from_numpy(mean).float().view(1, 80, 1)
    std = torch.from_numpy(std).float().view(1, 80, 1)
    return mean, std

def normalize_mel(mel, mean, std):
    mel = torch.from_numpy(mel).float().unsqueeze(0)
    mel = (mel - mean) / std
    return mel

def denormalize_mel(mel_norm, mean, std):
    return mel_norm * std + mean

def find_bnf_path(data_root, spk, mel_fname):
    bnf_dir = os.path.join(data_root, "bnf", spk)
    p1 = os.path.join(bnf_dir, mel_fname.replace(".npy", ".ling_feat.npy"))
    p2 = os.path.join(bnf_dir, mel_fname)
    if os.path.exists(p1):
        return p1
    if os.path.exists(p2):
        return p2
    raise FileNotFoundError(f"BNF not found for {spk}/{mel_fname}")

def load_sample_from_data(data_root, spk, mel_fname):
    mel_path = os.path.join(data_root, "mel", spk, mel_fname)
    bnf_path = find_bnf_path(data_root, spk, mel_fname)

    mel = ensure_mel_shape(np.load(mel_path))
    bnf = ensure_bnf_shape(np.load(bnf_path))
    bnf = resample_bnf_to_mel_len(bnf, mel.shape[1])

    return mel, bnf, mel_path, bnf_path

def parse_global_idx(fname):
    nums = re.findall(r'\d+', fname)
    local_idx = int(nums[-1])
    if 'arctic_b' in fname or '_b' in fname:
        return local_idx + 593
    return local_idx

def normalize_text(s):
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9\s']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [ ]:
# ===== Cell 7: 论文 closed-set evaluation utterances（每人 32 条） =====
CLOSEDSET_SPKS = ["clb", "bdl", "slt", "rms"]

def collect_eval_files_for_speaker(data_root, spk):
    mel_dir = os.path.join(data_root, "mel", spk)
    files = sorted([x for x in os.listdir(mel_dir) if x.endswith(".npy")])

    eval_files = []
    for f in files:
        idx = parse_global_idx(f)
        if 1101 <= idx <= 1132:
            eval_files.append(f)

    if len(eval_files) != 32:
        print(f"[Warning] {spk} eval count = {len(eval_files)} (expected 32)")
    return eval_files

eval_lists = {spk: collect_eval_files_for_speaker(DATA_ROOT, spk) for spk in CLOSEDSET_SPKS}

for spk, items in eval_lists.items():
    print(spk, len(items), items[:3], "...", items[-3:])

clb 32 ['arctic_b0508.npy', 'arctic_b0509.npy', 'arctic_b0510.npy'] ... ['arctic_b0537.npy', 'arctic_b0538.npy', 'arctic_b0539.npy']
bdl 32 ['arctic_b0508.npy', 'arctic_b0509.npy', 'arctic_b0510.npy'] ... ['arctic_b0537.npy', 'arctic_b0538.npy', 'arctic_b0539.npy']
slt 32 ['arctic_b0508.npy', 'arctic_b0509.npy', 'arctic_b0510.npy'] ... ['arctic_b0537.npy', 'arctic_b0538.npy', 'arctic_b0539.npy']
rms 32 ['arctic_b0508.npy', 'arctic_b0509.npy', 'arctic_b0510.npy'] ... ['arctic_b0537.npy', 'arctic_b0538.npy', 'arctic_b0539.npy']


In [ ]:
# ===== Cell 8: 12 个有方向的 closed-set cross-speaker 对 =====
PAIRS = []
for src_spk in CLOSEDSET_SPKS:
    for tgt_spk in CLOSEDSET_SPKS:
        if src_spk != tgt_spk:
            PAIRS.append((src_spk, tgt_spk))

print("Pairs =", len(PAIRS))
for p in PAIRS:
    print(p)
print("Expected number of generated wavs =", len(PAIRS) * 32)

Pairs = 12
('clb', 'bdl')
('clb', 'slt')
('clb', 'rms')
('bdl', 'clb')
('bdl', 'slt')
('bdl', 'rms')
('slt', 'clb')
('slt', 'bdl')
('slt', 'rms')
('rms', 'clb')
('rms', 'bdl')
('rms', 'slt')
Expected number of generated wavs = 384


In [ ]:
# ===== Cell 9: 说话人映射 =====
SPK2ID = {"clb": 0, "bdl": 1, "slt": 2, "rms": 3}
ID2SPK = {v: k for k, v in SPK2ID.items()}
print(SPK2ID)

{'clb': 0, 'bdl': 1, 'slt': 2, 'rms': 3}


In [ ]:
# ===== Cell 10: 构建模型并加载 checkpoint =====
def build_model_and_diffusion(device="cuda"):
    model = VoiceGrad(
        n_mels=80,
        n_bnf=144,
        n_channels=512,
        n_spk=4
    ).to(device)

    diffusion = VoiceGradDiffusion(
        n_levels=20,
        offset=0.008
    ).to(device)

    return model, diffusion

def load_ckpt_to_model(ckpt_path, model, map_location="cpu"):
    ckpt = torch.load(ckpt_path, map_location=map_location)

    if isinstance(ckpt, dict) and "model" in ckpt:
        state_dict = ckpt["model"]
        epoch = ckpt.get("epoch", "unknown")
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
        epoch = ckpt.get("epoch", "unknown")
    else:
        state_dict = ckpt
        epoch = "unknown"

    model.load_state_dict(state_dict, strict=True)
    return epoch, ckpt

model, diffusion = build_model_and_diffusion(device=DEVICE)
epoch, raw_ckpt = load_ckpt_to_model(CKPT_PATH, model, map_location=DEVICE)
model.eval()
print("Loaded checkpoint epoch =", epoch)

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loaded checkpoint epoch = 1000


In [ ]:
# ===== Cell 11: 加载 HiFi-GAN =====
hifigan_ready = False
generator = None
h = None

try:
    if HIFIGAN_ROOT not in sys.path:
        sys.path.append(HIFIGAN_ROOT)

    from env import AttrDict
    from models import Generator

    with open(HIFIGAN_CONFIG, "r") as f:
        h = AttrDict(json.load(f))

    generator = Generator(h).to(DEVICE)
    ckpt = torch.load(HIFIGAN_CKPT, map_location=DEVICE)

    if isinstance(ckpt, dict) and "generator" in ckpt:
        generator.load_state_dict(ckpt["generator"])
    else:
        generator.load_state_dict(ckpt)

    generator.eval()
    generator.remove_weight_norm()

    hifigan_ready = True
    print("HiFi-GAN loaded OK")
except Exception as e:
    print("HiFi-GAN load failed:", e)
    print("请检查 HIFIGAN_CKPT 路径。")

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Removing weight norm...
HiFi-GAN loaded OK


In [ ]:
# ===== Cell 12: 推理参数 =====
START_LEVEL = 11   # 论文 DPM 版 Algorithm 4 的 L'
SAVE_MEL = False
SAVE_WAV = True

print("START_LEVEL =", START_LEVEL)
print("SAVE_WAV =", SAVE_WAV)
print("hifigan_ready =", hifigan_ready)
if not hifigan_ready and SAVE_WAV:
    raise RuntimeError("HiFi-GAN 未正确加载，无法继续生成 wav。")

START_LEVEL = 11
SAVE_WAV = True
hifigan_ready = True


In [ ]:
# ===== Cell 13: 单条转换 =====
@torch.no_grad()
def convert_one(src_spk, src_mel_file, tgt_spk, mel_mean, mel_std):
    mel_np, bnf_np, mel_path, bnf_path = load_sample_from_data(DATA_ROOT, src_spk, src_mel_file)

    mel_norm = normalize_mel(mel_np, mel_mean, mel_std).to(DEVICE)         # [1,80,T]
    bnf_tensor = torch.from_numpy(bnf_np).float().unsqueeze(0).to(DEVICE)  # [1,144,T]
    target_id = torch.tensor([SPK2ID[tgt_spk]], dtype=torch.long, device=DEVICE)

    converted_norm = diffusion.sample(
        model=model,
        x_source=mel_norm,
        speaker_id=target_id,
        bnf=bnf_tensor,
        start_level=START_LEVEL
    )

    converted_mel = denormalize_mel(converted_norm.cpu(), mel_mean, mel_std)  # [1,80,T]
    return converted_mel

In [ ]:
# ===== Cell 14: 全量 closed-set 12 对 × 32 条 = 384 条 wav =====
mel_mean, mel_std = load_stats(DATA_ROOT)

all_meta = []

for src_spk, tgt_spk in PAIRS:
    pair_dir = os.path.join(GEN_WAV_ROOT, f"{src_spk}_to_{tgt_spk}")
    os.makedirs(pair_dir, exist_ok=True)

    src_files = eval_lists[src_spk]
    print(f"Running pair: {src_spk} -> {tgt_spk} | n={len(src_files)}")

    for src_mel_file in tqdm(src_files, leave=False):
        base = src_mel_file.replace(".npy", "")
        converted_mel = convert_one(src_spk, src_mel_file, tgt_spk, mel_mean, mel_std)

        mel_out_path = None
        if SAVE_MEL:
            mel_out_path = os.path.join(pair_dir, f"{base}.converted_mel.npy")
            np.save(mel_out_path, converted_mel.squeeze(0).numpy(), allow_pickle=False)

        wav_out_path = None
        if SAVE_WAV:
            y_g_hat = generator(converted_mel.to(DEVICE))
            wav = y_g_hat.squeeze().detach().cpu().numpy()
            wav_out_path = os.path.join(pair_dir, f"{base}.wav")
            sf.write(wav_out_path, wav, h.sampling_rate)

        all_meta.append({
            "src_spk": src_spk,
            "tgt_spk": tgt_spk,
            "utt_id": base,
            "mel_out_path": mel_out_path,
            "wav_out_path": wav_out_path
        })

meta_df = pd.DataFrame(all_meta)
meta_csv = os.path.join(OUT_ROOT, "generated_meta.csv")
meta_df.to_csv(meta_csv, index=False, encoding="utf-8-sig")

print("Done. Total generated =", len(all_meta))
print("Meta saved to:", meta_csv)

expected = len(PAIRS) * 32
if len(all_meta) != expected:
    print(f"[Warning] expected {expected}, but got {len(all_meta)}")
else:
    print("Count check OK.")

Running pair: clb -> bdl | n=32


Running pair: clb -> slt | n=32


Running pair: clb -> rms | n=32


Running pair: bdl -> clb | n=32


Running pair: bdl -> slt | n=32


...

Running pair: slt -> rms | n=32


Running pair: rms -> clb | n=32


Running pair: rms -> bdl | n=32


Running pair: rms -> slt | n=32


Done. Total generated = 384
Meta saved to: /content/drive/MyDrive/VoiceGrad/ckpt1000_closedset_cer_paper_protocol/generated_meta.csv
Count check OK.


In [ ]:
# ===== Cell 15: 如果还没填 TEXT_PATH，帮你找 transcript 候选 =====
candidates = []
for root, dirs, files in os.walk(PROJECT_ROOT):
    for f in files:
        lower = f.lower()
        if lower.endswith(".txt") or lower.endswith(".data"):
            if "txt.done" in lower or "prompt" in lower or "trans" in lower or "sentence" in lower or "arctic" in lower:
                candidates.append(os.path.join(root, f))

print("Transcript candidates:")
for x in candidates[:100]:
    print(" ", x)

In [ ]:
# ===== Cell 16: 解析 transcript =====
def load_ref_map(text_path):
    if text_path is None:
        raise ValueError("TEXT_PATH is None. 先在 Cell 3 填好 transcript 路径。")

    ref_map = {}

    with open(text_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # CMU ARCTIC txt.done.data 常见格式:
        # ( arctic_a0001 "TEXT ..." )
        m = re.match(r'\(\s*(arctic_[ab]\d+)\s+"(.+)"\s*\)', line)
        if m:
            utt_id = m.group(1)
            text = m.group(2)
            ref_map[utt_id] = normalize_text(text)
            continue

        # 通用格式：utt|text
        if "|" in line:
            parts = line.split("|", 1)
            utt_id = parts[0].strip()
            text = parts[1].strip()
            ref_map[utt_id] = normalize_text(text)
            continue

        # 通用格式：utt\ttext
        if "\t" in line:
            parts = line.split("\t", 1)
            utt_id = parts[0].strip()
            text = parts[1].strip()
            ref_map[utt_id] = normalize_text(text)
            continue

    if len(ref_map) == 0:
        raise RuntimeError("Failed to parse transcript file. 请检查 TEXT_PATH 和文件格式。")

    return ref_map

ref_map = load_ref_map(TEXT_PATH)
print("Loaded refs:", len(ref_map))
for k in list(ref_map.keys())[:5]:
    print(k, "=>", ref_map[k][:80])

Loaded refs: 1132
arctic_a0001 => author of the danger trail philip steels etc
arctic_a0002 => not at this particular case tom apologized whittemore
arctic_a0003 => for the twentieth time that evening the two men shook hands
arctic_a0004 => lord but i'm glad to see you again phil
arctic_a0005 => will we ever forget it


In [ ]:
# ===== Cell 17: 加载论文同口径 ASR：wav2vec 2.0 Large LV-60K =====
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

ASR_MODEL_NAME = "facebook/wav2vec2-large-960h-lv60-self"

processor = Wav2Vec2Processor.from_pretrained(ASR_MODEL_NAME)
asr_model = Wav2Vec2ForCTC.from_pretrained(ASR_MODEL_NAME).to(DEVICE)
asr_model.eval()

print("Loaded ASR model:", ASR_MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/162 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/423 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-large-960h-lv60-self
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loaded ASR model: facebook/wav2vec2-large-960h-lv60-self


In [ ]:
# ===== Cell 18: 单条 ASR 转写 =====
@torch.no_grad()
def transcribe_wav_ctc(wav_path, target_sr=16000):
    wav, sr = sf.read(wav_path)
    if wav.ndim == 2:
        wav = wav.mean(axis=1)
    if sr != target_sr:
        wav = librosa.resample(wav.astype(np.float32), orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    inputs = processor(
        wav,
        sampling_rate=sr,
        return_tensors="pt",
        padding=True
    )

    input_values = inputs.input_values.to(DEVICE)
    attention_mask = inputs.attention_mask.to(DEVICE) if "attention_mask" in inputs else None

    logits = asr_model(input_values, attention_mask=attention_mask).logits
    pred_ids = torch.argmax(logits, dim=-1)

    text = processor.batch_decode(pred_ids)[0]
    return normalize_text(text)

In [ ]:
# ===== Cell 19: 计算 384 条 wav 的 CER =====
from jiwer import cer

rows = []

for item in tqdm(all_meta):
    src_spk = item["src_spk"]
    tgt_spk = item["tgt_spk"]
    utt_id = item["utt_id"]
    wav_path = item["wav_out_path"]

    if utt_id not in ref_map:
        print("[Skip no ref]", utt_id)
        continue

    ref = ref_map[utt_id]
    hyp = transcribe_wav_ctc(wav_path)
    cur_cer = cer(ref, hyp)

    rows.append({
        "src_spk": src_spk,
        "tgt_spk": tgt_spk,
        "utt_id": utt_id,
        "ref": ref,
        "hyp": hyp,
        "cer": cur_cer,
        "wav_path": wav_path
    })

cer_df = pd.DataFrame(rows)
detail_csv = os.path.join(OUT_ROOT, "cer_detail.csv")
cer_df.to_csv(detail_csv, index=False, encoding="utf-8-sig")

print("rows =", len(cer_df))
print("detail saved to:", detail_csv)
cer_df.head()

100%|██████████| 384/384 [00:20<00:00, 18.49it/s]

rows = 384
detail saved to: /content/drive/MyDrive/VoiceGrad/ckpt1000_closedset_cer_paper_protocol/cer_detail.csv


,src_spk,tgt_spk,utt_id,ref,hyp,cer,wav_path
0,clb,bdl,arctic_b0508,in his anxiety and solicitude and love they di...,in his anxiety and solicitude and love they di...,0.000000,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...
1,clb,bdl,arctic_b0509,he had fulfilled his duty and paid properly,he had fulfilled his duty and paid properly,0.000000,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...
2,clb,bdl,arctic_b0510,he knew what taboos he was violating,he knew what taboos he was violating,0.000000,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...
3,clb,bdl,arctic_b0511,do you value your hide,do you value your hide,0.000000,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...
4,clb,bdl,arctic_b0512,you should have seen them when they heard me s...,you should have seen them when they heard me s...,0.032787,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...


In [ ]:
# ===== Cell 20: 按论文表格顺序汇总 12 对 + All pairs =====
pair_order = [
    ("bdl", "clb"), ("bdl", "slt"), ("bdl", "rms"),
    ("clb", "bdl"), ("clb", "slt"), ("clb", "rms"),
    ("slt", "clb"), ("slt", "bdl"), ("slt", "rms"),
    ("rms", "clb"), ("rms", "bdl"), ("rms", "slt"),
]

summary_rows = []
for src_spk, tgt_spk in pair_order:
    sub = cer_df[(cer_df["src_spk"] == src_spk) & (cer_df["tgt_spk"] == tgt_spk)]
    summary_rows.append({
        "src_spk": src_spk,
        "tgt_spk": tgt_spk,
        "count": int(len(sub)),
        "cer_mean_pct": float(sub["cer"].mean() * 100.0),
        "cer_std_pct": float(sub["cer"].std(ddof=0) * 100.0)
    })

summary_pair = pd.DataFrame(summary_rows)
summary_pair_csv = os.path.join(OUT_ROOT, "cer_summary_by_pair.csv")
summary_pair.to_csv(summary_pair_csv, index=False, encoding="utf-8-sig")

all_pairs = {
    "count": int(len(cer_df)),
    "cer_mean_pct": float(cer_df["cer"].mean() * 100.0),
    "cer_std_pct": float(cer_df["cer"].std(ddof=0) * 100.0),
    "cer_median_pct": float(cer_df["cer"].median() * 100.0),
}

print("===== CER by pair (paper order) =====")
display(summary_pair)

print("\n===== CER all pairs =====")
print(all_pairs)

print("\nSaved pair summary to:", summary_pair_csv)

===== CER by pair (paper order) =====


,src_spk,tgt_spk,count,cer_mean_pct,cer_std_pct
0,bdl,clb,32,2.274614,5.574544
1,bdl,slt,32,2.486193,5.681399
2,bdl,rms,32,3.050909,6.081313
3,clb,bdl,32,1.439496,2.326435
4,clb,slt,32,2.045743,2.819005
5,clb,rms,32,1.378448,2.105272
6,slt,clb,32,1.642704,4.208781
7,slt,bdl,32,2.363803,6.572615
8,slt,rms,32,1.679621,2.508610
9,rms,clb,32,1.728929,4.209352



===== CER all pairs =====
{'count': 384, 'cer_mean_pct': 1.9550883004932647, 'cer_std_pct': 4.4360461828296645, 'cer_median_pct': 0.0}

Saved pair summary to: /content/drive/MyDrive/VoiceGrad/ckpt1000_closedset_cer_paper_protocol/cer_summary_by_pair.csv


In [ ]:
# ===== Cell 21: 生成一个接近论文表 VI(a) 的打印版本 =====
print("CER [%] COMPARISONS (Your VoiceGrad checkpoint)")
print()
for src_spk in ["bdl", "clb", "slt", "rms"]:
    sub = summary_pair[summary_pair["src_spk"] == src_spk]
    print(src_spk)
    for _, row in sub.iterrows():
        print(
            f"  {row['tgt_spk']}: "
            f"{row['cer_mean_pct']:.2f} ± {row['cer_std_pct']:.2f} "
            f"(n={int(row['count'])})"
        )
    print()

print(
    f"All pairs: {all_pairs['cer_mean_pct']:.2f} ± {all_pairs['cer_std_pct']:.2f} "
    f"(n={all_pairs['count']})"
)

CER [%] COMPARISONS (Your VoiceGrad checkpoint)

bdl
  clb: 2.27 ± 5.57 (n=32)
  slt: 2.49 ± 5.68 (n=32)
  rms: 3.05 ± 6.08 (n=32)

clb
  bdl: 1.44 ± 2.33 (n=32)
  slt: 2.05 ± 2.82 (n=32)
  rms: 1.38 ± 2.11 (n=32)

slt
  clb: 1.64 ± 4.21 (n=32)
  bdl: 2.36 ± 6.57 (n=32)
  rms: 1.68 ± 2.51 (n=32)

rms
  clb: 1.73 ± 4.21 (n=32)
  bdl: 1.68 ± 4.17 (n=32)
  slt: 1.70 ± 3.60 (n=32)

All pairs: 1.96 ± 4.44 (n=384)


In [ ]:
# ===== Cell 22: 保存最终总表 =====
final_txt = os.path.join(OUT_ROOT, "cer_summary_final.txt")
with open(final_txt, "w", encoding="utf-8") as f:
    f.write("CER [%] COMPARISONS (Your VoiceGrad checkpoint)\n\n")
    for src_spk in ["bdl", "clb", "slt", "rms"]:
        sub = summary_pair[summary_pair["src_spk"] == src_spk]
        f.write(f"{src_spk}\n")
        for _, row in sub.iterrows():
            f.write(
                f"  {row['tgt_spk']}: "
                f"{row['cer_mean_pct']:.2f} ± {row['cer_std_pct']:.2f} "
                f"(n={int(row['count'])})\n"
            )
        f.write("\n")
    f.write(
        f"All pairs: {all_pairs['cer_mean_pct']:.2f} ± {all_pairs['cer_std_pct']:.2f} "
        f"(n={all_pairs['count']})\n"
    )

print("Saved final summary to:", final_txt)
print("OUT_ROOT =", OUT_ROOT)

Saved final summary to: /content/drive/MyDrive/VoiceGrad/ckpt1000_closedset_cer_paper_protocol/cer_summary_final.txt
OUT_ROOT = /content/drive/MyDrive/VoiceGrad/ckpt1000_closedset_cer_paper_protocol


MCD,LFC,pMOS

In [ ]:
!pip -q install pyworld pysptk librosa soundfile pandas tqdm scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 24.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.9/461.9 kB 35.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/VoiceGrad"

# 你前一个 CER notebook 生成的 384 条 wav 根目录
GENERATED_WAV_ROOT = os.path.join(PROJECT_ROOT, "ckpt1000_closedset_cer_paper_protocol", "generated_wavs")

# 原始 target 真值 wav 根目录
# 常见结构：data/wav/{spk}/wav/*.wav
REF_WAV_ROOT = os.path.join(PROJECT_ROOT, "data", "wav")

OUT_ROOT = os.path.join(PROJECT_ROOT, "ckpt1000_closedset_mcd_lfc_pmos")
os.makedirs(OUT_ROOT, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("GENERATED_WAV_ROOT =", GENERATED_WAV_ROOT)
print("REF_WAV_ROOT =", REF_WAV_ROOT)
print("OUT_ROOT =", OUT_ROOT)
print("generated exists?", os.path.isdir(GENERATED_WAV_ROOT))
print("ref exists?", os.path.isdir(REF_WAV_ROOT))

PROJECT_ROOT = /content/drive/MyDrive/VoiceGrad
GENERATED_WAV_ROOT = /content/drive/MyDrive/VoiceGrad/ckpt1000_closedset_cer_paper_protocol/generated_wavs
REF_WAV_ROOT = /content/drive/MyDrive/VoiceGrad/data/wav
OUT_ROOT = /content/drive/MyDrive/VoiceGrad/ckpt1000_closedset_mcd_lfc_pmos
generated exists? True
ref exists? True


In [ ]:
import os, re, glob
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import pyworld as pw
import pysptk
from tqdm import tqdm
from scipy.spatial.distance import cdist
from librosa.sequence import dtw

SR = 16000
MCEP_ORDER = 24
MCEP_ALPHA = 0.42
FRAME_PERIOD_MS = 10.0

PAIR_ORDER = [
    ("bdl", "clb"), ("bdl", "slt"), ("bdl", "rms"),
    ("clb", "bdl"), ("clb", "slt"), ("clb", "rms"),
    ("slt", "clb"), ("slt", "bdl"), ("slt", "rms"),
    ("rms", "clb"), ("rms", "bdl"), ("rms", "slt"),
]

In [ ]:
def collect_generated_wavs(root):
    rows = []
    for pair_dir in sorted(glob.glob(os.path.join(root, "*_to_*"))):
        m = re.match(r"([a-z]+)_to_([a-z]+)", os.path.basename(pair_dir))
        if not m:
            continue
        src_spk, tgt_spk = m.group(1), m.group(2)

        for wav_path in sorted(glob.glob(os.path.join(pair_dir, "*.wav"))):
            utt_id = os.path.splitext(os.path.basename(wav_path))[0]
            rows.append({
                "src_spk": src_spk,
                "tgt_spk": tgt_spk,
                "utt_id": utt_id,
                "gen_wav_path": wav_path,
            })
    return pd.DataFrame(rows)

def build_ref_index(root):
    idx = {}
    for spk_dir in sorted(glob.glob(os.path.join(root, "*"))):
        spk = os.path.basename(spk_dir)
        if not os.path.isdir(spk_dir):
            continue

        files = glob.glob(os.path.join(spk_dir, "wav", "*.wav"))
        files += glob.glob(os.path.join(spk_dir, "*.wav"))
        if not files:
            files = glob.glob(os.path.join(spk_dir, "**", "*.wav"), recursive=True)

        for wav_path in files:
            utt_id = os.path.splitext(os.path.basename(wav_path))[0]
            idx[(spk, utt_id)] = wav_path
    return idx

gen_df = collect_generated_wavs(GENERATED_WAV_ROOT)
ref_index = build_ref_index(REF_WAV_ROOT)

gen_df["ref_wav_path"] = [
    ref_index.get((r["tgt_spk"], r["utt_id"]), None)
    for _, r in gen_df.iterrows()
]

print("generated wav count:", len(gen_df), "expected:", 384)
print("missing refs:", gen_df["ref_wav_path"].isna().sum())
display(gen_df.head())

generated wav count: 384 expected: 384
missing refs: 0


,src_spk,tgt_spk,utt_id,gen_wav_path,ref_wav_path
0,bdl,clb,arctic_b0508,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...,/content/drive/MyDrive/VoiceGrad/data/wav/clb/...
1,bdl,clb,arctic_b0509,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...,/content/drive/MyDrive/VoiceGrad/data/wav/clb/...
2,bdl,clb,arctic_b0510,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...,/content/drive/MyDrive/VoiceGrad/data/wav/clb/...
3,bdl,clb,arctic_b0511,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...,/content/drive/MyDrive/VoiceGrad/data/wav/clb/...
4,bdl,clb,arctic_b0512,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...,/content/drive/MyDrive/VoiceGrad/data/wav/clb/...


In [ ]:
def load_wav_mono_16k(wav_path, sr=SR):
    wav, file_sr = sf.read(wav_path)
    if wav.ndim == 2:
        wav = wav.mean(axis=1)
    wav = wav.astype(np.float64)
    if file_sr != sr:
        wav = librosa.resample(
            wav.astype(np.float32),
            orig_sr=file_sr,
            target_sr=sr
        ).astype(np.float64)
    return wav

def extract_f0_mcep(wav_path):
    wav = load_wav_mono_16k(wav_path)

    f0, timeaxis = pw.harvest(
        wav,
        SR,
        frame_period=FRAME_PERIOD_MS,
        f0_floor=20.0,
        f0_ceil=600.0
    )
    sp = pw.cheaptrick(wav, f0, timeaxis, SR)
    mcep = pysptk.sp2mc(sp, order=MCEP_ORDER, alpha=MCEP_ALPHA)

    return {
        "f0": f0.astype(np.float64),
        "mcep": mcep.astype(np.float64),
    }

MCD_CONST = 10.0 / np.log(10.0) * np.sqrt(2.0)

def score_one_mcd_lfc(gen_wav_path, ref_wav_path):
    g = extract_f0_mcep(gen_wav_path)
    r = extract_f0_mcep(ref_wav_path)

    C = cdist(g["mcep"][:, 1:], r["mcep"][:, 1:], metric="euclidean")
    _, wp = dtw(C=C)
    wp = np.array(wp[::-1], dtype=np.int64)

    dists = []
    xs, ys = [], []

    for i, j in wp:
        diff = g["mcep"][i, 1:] - r["mcep"][j, 1:]
        dists.append(MCD_CONST * np.sqrt(np.sum(diff * diff)))

        if g["f0"][i] > 0 and r["f0"][j] > 0:
            xs.append(np.log(g["f0"][i]))
            ys.append(np.log(r["f0"][j]))

    mcd = float(np.mean(dists))

    xs = np.asarray(xs)
    ys = np.asarray(ys)
    if len(xs) < 2 or np.std(xs) < 1e-12 or np.std(ys) < 1e-12:
        lfc = np.nan
    else:
        lfc = float(np.corrcoef(xs, ys)[0, 1])

    return mcd, lfc

In [ ]:
rows = []
valid_df = gen_df.dropna(subset=["ref_wav_path"]).reset_index(drop=True)

for _, row in tqdm(valid_df.iterrows(), total=len(valid_df)):
    try:
        mcd, lfc = score_one_mcd_lfc(row["gen_wav_path"], row["ref_wav_path"])
    except Exception as e:
        print("[Error]", row["gen_wav_path"], e)
        mcd, lfc = np.nan, np.nan

    rows.append({
        **row.to_dict(),
        "mcd_db": mcd,
        "lfc": lfc,
    })

mcd_lfc_df = pd.DataFrame(rows)
mcd_lfc_df.to_csv(os.path.join(OUT_ROOT, "mcd_lfc_detail.csv"), index=False, encoding="utf-8-sig")
display(mcd_lfc_df.head())

100%|██████████| 384/384 [36:24<00:00,  5.69s/it]


,src_spk,tgt_spk,utt_id,gen_wav_path,ref_wav_path,mcd_db,lfc
0,bdl,clb,arctic_b0508,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...,/content/drive/MyDrive/VoiceGrad/data/wav/clb/...,5.735648,0.722436
1,bdl,clb,arctic_b0509,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...,/content/drive/MyDrive/VoiceGrad/data/wav/clb/...,6.246879,0.028759
2,bdl,clb,arctic_b0510,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...,/content/drive/MyDrive/VoiceGrad/data/wav/clb/...,6.016983,0.209302
3,bdl,clb,arctic_b0511,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...,/content/drive/MyDrive/VoiceGrad/data/wav/clb/...,5.271260,0.374262
4,bdl,clb,arctic_b0512,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...,/content/drive/MyDrive/VoiceGrad/data/wav/clb/...,6.354345,0.368658


In [ ]:
def mean_ci95(arr):
    arr = pd.Series(arr).dropna().to_numpy(dtype=np.float64)
    n = len(arr)
    if n == 0:
        return np.nan, np.nan, 0
    mean = float(np.mean(arr))
    std = float(np.std(arr, ddof=0))
    ci95 = float(1.96 * std / np.sqrt(n))
    return mean, ci95, n

pair_rows = []
for src_spk, tgt_spk in PAIR_ORDER:
    sub = mcd_lfc_df[
        (mcd_lfc_df["src_spk"] == src_spk) &
        (mcd_lfc_df["tgt_spk"] == tgt_spk)
    ]

    mcd_mean, mcd_ci95, mcd_n = mean_ci95(sub["mcd_db"])
    lfc_mean, lfc_ci95, lfc_n = mean_ci95(sub["lfc"])

    pair_rows.append({
        "src_spk": src_spk,
        "tgt_spk": tgt_spk,
        "count": int(len(sub)),
        "mcd_mean_db": mcd_mean,
        "mcd_ci95_db": mcd_ci95,
        "lfc_mean": lfc_mean,
        "lfc_ci95": lfc_ci95,
        "valid_mcd_n": mcd_n,
        "valid_lfc_n": lfc_n,
    })

pair_summary = pd.DataFrame(pair_rows)
pair_summary.to_csv(os.path.join(OUT_ROOT, "mcd_lfc_summary_by_pair.csv"), index=False, encoding="utf-8-sig")

all_mcd_mean, all_mcd_ci95, _ = mean_ci95(mcd_lfc_df["mcd_db"])
all_lfc_mean, all_lfc_ci95, _ = mean_ci95(mcd_lfc_df["lfc"])

display(pair_summary)
print({
    "all_pairs_mcd_db": all_mcd_mean,
    "all_pairs_mcd_ci95_db": all_mcd_ci95,
    "all_pairs_lfc": all_lfc_mean,
    "all_pairs_lfc_ci95": all_lfc_ci95,
})

,src_spk,tgt_spk,count,mcd_mean_db,mcd_ci95_db,lfc_mean,lfc_ci95,valid_mcd_n,valid_lfc_n
0,bdl,clb,32,6.044336,0.134675,0.311626,0.084941,32,32
1,bdl,slt,32,6.141156,0.086062,0.458885,0.093924,32,32
2,bdl,rms,32,6.044550,0.095994,0.479214,0.074358,32,32
3,clb,bdl,32,6.244711,0.105604,0.437976,0.112016,32,32
4,clb,slt,32,6.116989,0.068820,0.675992,0.068627,32,32
5,clb,rms,32,5.956829,0.073672,0.495384,0.099794,32,32
6,slt,clb,32,5.774147,0.082077,0.567661,0.093348,32,32
7,slt,bdl,32,6.264133,0.102946,0.325934,0.139930,32,32
8,slt,rms,32,5.964470,0.076635,0.512945,0.082573,32,32
9,rms,clb,32,6.031425,0.077649,0.476671,0.089072,32,32


{'all_pairs_mcd_db': 6.107304456350231, 'all_pairs_mcd_ci95_db': 0.031059556884040846, 'all_pairs_lfc': 0.46751753737515395, 'all_pairs_lfc_ci95': 0.0302229537963966}


In [ ]:
!pip -q install speechmos onnxruntime librosa soundfile pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 134.6 MB/s eta 0:00:00


In [ ]:
import os, re, glob, numpy as np, pandas as pd, librosa
from tqdm import tqdm
from speechmos import dnsmos

PROJECT_ROOT = "/content/drive/MyDrive/VoiceGrad"
GENERATED_WAV_ROOT = os.path.join(PROJECT_ROOT, "ckpt1000_closedset_cer_paper_protocol", "generated_wavs")
OUT_ROOT = os.path.join(PROJECT_ROOT, "ckpt1000_closedset_mcd_lfc_pmos")
os.makedirs(OUT_ROOT, exist_ok=True)

PAIR_ORDER = [
    ("bdl", "clb"), ("bdl", "slt"), ("bdl", "rms"),
    ("clb", "bdl"), ("clb", "slt"), ("clb", "rms"),
    ("slt", "clb"), ("slt", "bdl"), ("slt", "rms"),
    ("rms", "clb"), ("rms", "bdl"), ("rms", "slt"),
]

print("GENERATED_WAV_ROOT =", GENERATED_WAV_ROOT)
print("OUT_ROOT =", OUT_ROOT)
print("exists? ", os.path.isdir(GENERATED_WAV_ROOT))

GENERATED_WAV_ROOT = /content/drive/MyDrive/VoiceGrad/ckpt1000_closedset_cer_paper_protocol/generated_wavs
OUT_ROOT = /content/drive/MyDrive/VoiceGrad/ckpt1000_closedset_mcd_lfc_pmos
exists?  True


In [ ]:
def collect_generated_wavs(root):
    rows = []
    for pair_dir in sorted(glob.glob(os.path.join(root, "*_to_*"))):
        m = re.match(r"([a-z]+)_to_([a-z]+)", os.path.basename(pair_dir))
        if not m:
            continue
        src_spk, tgt_spk = m.group(1), m.group(2)

        for wav_path in sorted(glob.glob(os.path.join(pair_dir, "*.wav"))):
            utt_id = os.path.splitext(os.path.basename(wav_path))[0]
            rows.append({
                "src_spk": src_spk,
                "tgt_spk": tgt_spk,
                "utt_id": utt_id,
                "wav_path": wav_path,
            })
    return pd.DataFrame(rows)

wav_df = collect_generated_wavs(GENERATED_WAV_ROOT)
print("wav count =", len(wav_df), "expected =", 384)
display(wav_df.head())

wav count = 384 expected = 384


,src_spk,tgt_spk,utt_id,wav_path
0,bdl,clb,arctic_b0508,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...
1,bdl,clb,arctic_b0509,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...
2,bdl,clb,arctic_b0510,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...
3,bdl,clb,arctic_b0511,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...
4,bdl,clb,arctic_b0512,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...


In [ ]:
rows = []

for _, row in tqdm(wav_df.iterrows(), total=len(wav_df)):
    wav_path = row["wav_path"]
    try:
        audio, sr = librosa.load(wav_path, sr=16000)
        scores = dnsmos.run(audio, sr)
        pmos_score = float(scores["ovrl_mos"])
    except Exception as e:
        print("[DNSMOS error]", wav_path, "=>", e)
        pmos_score = np.nan

    rows.append({
        "src_spk": row["src_spk"],
        "tgt_spk": row["tgt_spk"],
        "utt_id": row["utt_id"],
        "wav_path": wav_path,
        "pmos": pmos_score,
    })

pmos_df = pd.DataFrame(rows)

detail_csv = os.path.join(OUT_ROOT, "pmos_dnsmos_detail.csv")
pmos_df.to_csv(detail_csv, index=False, encoding="utf-8-sig")

print("saved:", detail_csv)
display(pmos_df.head())
print("all_pairs_mean =", float(pmos_df["pmos"].mean()))
print("all_pairs_std  =", float(pmos_df["pmos"].std()))

100%|██████████| 384/384 [05:02<00:00,  1.27it/s]

saved: /content/drive/MyDrive/VoiceGrad/ckpt1000_closedset_mcd_lfc_pmos/pmos_dnsmos_detail.csv


,src_spk,tgt_spk,utt_id,wav_path,pmos
0,bdl,clb,arctic_b0508,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...,3.232661
1,bdl,clb,arctic_b0509,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...,3.179086
2,bdl,clb,arctic_b0510,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...,3.114292
3,bdl,clb,arctic_b0511,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...,3.087362
4,bdl,clb,arctic_b0512,/content/drive/MyDrive/VoiceGrad/ckpt1000_clos...,2.810636


all_pairs_mean = 3.200673244500121
all_pairs_std  = 0.18404789768792365


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
VoiceGrad closed-set checkpoint sweep evaluation (exclude ckpt1000)

作用：
1. 对多个 checkpoint 逐个做 closed-set 全量推理（12 个方向 × 32 条 = 384 条）
2. 用和你之前 1000 ckpt notebook 相同的方法评测 CER / MCD / LFC / pMOS
3. 每个 checkpoint 单独保存生成 wav 和评测结果，避免和 ckpt1000 冲突
4. 最后汇总成一个总表，方便直接比较 500/1500/2000/2500/3000/3500/4000/best

默认输出目录：
    {PROJECT_ROOT}/ckpt_sweep_closedset_eval_except1000/

你只需要先改顶部配置区的路径，然后运行：
    python voicegrad_closedset_ckpt_sweep_eval.py

如果你在 Colab 里跑，也可以直接把这个脚本内容贴进一个 cell 运行。
"""

import gc
import glob
import json
import math
import os
import re
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import torch.nn.functional as F
from tqdm import tqdm


# ============================================================
# 0. 配置区：先改这里
# ============================================================
PROJECT_ROOT = "/content/drive/MyDrive/VoiceGrad"
DATA_ROOT = os.path.join(PROJECT_ROOT, "data")
CKPT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")

# HiFi-GAN
HIFIGAN_ROOT = os.path.join(PROJECT_ROOT, "hifi-gan")
HIFIGAN_CONFIG = os.path.join(HIFIGAN_ROOT, "config_16k.json")
HIFIGAN_CKPT = os.path.join(HIFIGAN_ROOT, "g_00105000")

# transcript
TEXT_PATH = os.path.join(DATA_ROOT, "cmuarctic.data.txt")

# 总输出目录：和 1000 ckpt 完全分开
RUN_NAME = "ckpt_sweep_closedset_eval_except1000"
OUT_BASE = os.path.join(PROJECT_ROOT, RUN_NAME)

# 要评测的 ckpt（明确排除 1000）
EVAL_CKPTS = [500, 1500, 2000, 2500, 3000, 3500, 4000, "best"]

# reverse diffusion 起点，沿用你之前 notebook
START_LEVEL = 11

# 是否跳过已经跑完的 checkpoint
SKIP_IF_DONE = True

# 如果某个 checkpoint 已经生成过 wav，是否复用已有 wav，不重新推理
REUSE_EXISTING_WAVS = True

# pMOS 仍然沿用你之前 notebook 的 speechmos.dnsmos 方案
# 注意：这严格来说不是论文原版 UTMOS，而是“和你 1000ckpt notebook 相同的方法”
USE_DNSMOS_PMOS = True

# 说话人映射（closed-set）
CLOSEDSET_SPKS = ["clb", "bdl", "slt", "rms"]
SPK2ID = {"clb": 0, "bdl": 1, "slt": 2, "rms": 3}

PAIR_ORDER = [
    ("bdl", "clb"), ("bdl", "slt"), ("bdl", "rms"),
    ("clb", "bdl"), ("clb", "slt"), ("clb", "rms"),
    ("slt", "clb"), ("slt", "bdl"), ("slt", "rms"),
    ("rms", "clb"), ("rms", "bdl"), ("rms", "slt"),
]

ASR_MODEL_NAME = "facebook/wav2vec2-large-960h-lv60-self"

# MCD/LFC 参数：沿用你 1000 notebook
SR = 16000
MCEP_ORDER = 24
MCEP_ALPHA = 0.42
FRAME_PERIOD_MS = 10.0
MCD_CONST = 10.0 / np.log(10.0) * np.sqrt(2.0)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ============================================================
# 1. 环境准备
# ============================================================
def prepare_project_imports(project_root: str) -> None:
    os.chdir(project_root)
    if project_root not in sys.path:
        sys.path.append(project_root)


def print_config() -> None:
    print("=" * 80)
    print("PROJECT_ROOT :", PROJECT_ROOT)
    print("DATA_ROOT    :", DATA_ROOT)
    print("CKPT_DIR     :", CKPT_DIR)
    print("HIFIGAN_ROOT :", HIFIGAN_ROOT)
    print("TEXT_PATH    :", TEXT_PATH)
    print("OUT_BASE     :", OUT_BASE)
    print("DEVICE       :", DEVICE)
    print("EVAL_CKPTS   :", EVAL_CKPTS)
    print("START_LEVEL  :", START_LEVEL)
    print("=" * 80)


def ensure_paths() -> None:
    required_dirs = [PROJECT_ROOT, DATA_ROOT, CKPT_DIR, HIFIGAN_ROOT]
    required_files = [HIFIGAN_CONFIG, HIFIGAN_CKPT, TEXT_PATH]

    for p in required_dirs:
        if not os.path.isdir(p):
            raise FileNotFoundError(f"目录不存在: {p}")
    for p in required_files:
        if not os.path.exists(p):
            raise FileNotFoundError(f"文件不存在: {p}")

    os.makedirs(OUT_BASE, exist_ok=True)


# ============================================================
# 2. 从项目里导入你当前使用的 VoiceGrad 代码
# ============================================================
def import_project_modules():
    from model import VoiceGrad  # type: ignore
    from diffusion import VoiceGradDiffusion  # type: ignore
    return VoiceGrad, VoiceGradDiffusion


# ============================================================
# 3. 通用函数（和你原 notebook 保持一致）
# ============================================================
def ensure_mel_shape(mel: np.ndarray) -> np.ndarray:
    if mel.ndim != 2:
        raise ValueError(f"mel shape wrong: {mel.shape}")
    if mel.shape[0] == 80:
        return mel
    if mel.shape[1] == 80:
        return mel.T
    raise ValueError(f"mel shape wrong: {mel.shape}")


def ensure_bnf_shape(bnf: np.ndarray) -> np.ndarray:
    if bnf.ndim != 2:
        raise ValueError(f"bnf shape wrong: {bnf.shape}")
    if bnf.shape[0] == 144:
        return bnf
    if bnf.shape[1] == 144:
        return bnf.T
    raise ValueError(f"bnf shape wrong: {bnf.shape}")


def resample_bnf_to_mel_len(bnf: np.ndarray, mel_len: int) -> np.ndarray:
    # 与你 notebook / dataset 对齐：沿时间轴线性插值到 mel 长度
    bnf_tensor = torch.from_numpy(bnf).float().unsqueeze(0)
    bnf_tensor = F.interpolate(
        bnf_tensor,
        size=mel_len,
        mode="linear",
        align_corners=False,
    )
    return bnf_tensor.squeeze(0).cpu().numpy()


def load_stats(data_root: str) -> Tuple[torch.Tensor, torch.Tensor]:
    mean = np.load(os.path.join(data_root, "stats", "mel_mean.npy"))
    std = np.load(os.path.join(data_root, "stats", "mel_std.npy"))
    mean_t = torch.from_numpy(mean).float().view(1, 80, 1)
    std_t = torch.from_numpy(std).float().view(1, 80, 1)
    return mean_t, std_t


def normalize_mel(mel: np.ndarray, mean: torch.Tensor, std: torch.Tensor) -> torch.Tensor:
    mel_t = torch.from_numpy(mel).float().unsqueeze(0)
    mel_t = (mel_t - mean) / std
    return mel_t


def denormalize_mel(mel_norm: torch.Tensor, mean: torch.Tensor, std: torch.Tensor) -> torch.Tensor:
    return mel_norm * std + mean


def find_bnf_path(data_root: str, spk: str, mel_fname: str) -> str:
    bnf_dir = os.path.join(data_root, "bnf", spk)
    p1 = os.path.join(bnf_dir, mel_fname.replace(".npy", ".ling_feat.npy"))
    p2 = os.path.join(bnf_dir, mel_fname)
    if os.path.exists(p1):
        return p1
    if os.path.exists(p2):
        return p2
    raise FileNotFoundError(f"BNF not found for {spk}/{mel_fname}")


def load_sample_from_data(data_root: str, spk: str, mel_fname: str):
    mel_path = os.path.join(data_root, "mel", spk, mel_fname)
    bnf_path = find_bnf_path(data_root, spk, mel_fname)

    mel = ensure_mel_shape(np.load(mel_path))
    bnf = ensure_bnf_shape(np.load(bnf_path))
    bnf = resample_bnf_to_mel_len(bnf, mel.shape[1])
    return mel, bnf, mel_path, bnf_path


def parse_global_idx(fname: str) -> int:
    nums = re.findall(r"\d+", fname)
    if not nums:
        raise ValueError(f"Cannot parse numeric index from: {fname}")
    local_idx = int(nums[-1])
    if "arctic_b" in fname or "_b" in fname:
        return local_idx + 593
    return local_idx


def normalize_text(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9\s']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


# ============================================================
# 4. 数据清单
# ============================================================
def collect_eval_files_for_speaker(data_root: str, spk: str) -> List[str]:
    mel_dir = os.path.join(data_root, "mel", spk)
    files = sorted([x for x in os.listdir(mel_dir) if x.endswith(".npy")])

    eval_files = []
    for f in files:
        idx = parse_global_idx(f)
        if 1101 <= idx <= 1132:
            eval_files.append(f)

    if len(eval_files) != 32:
        print(f"[Warning] {spk} eval count = {len(eval_files)} (expected 32)")
    return eval_files


def get_closedset_eval_lists(data_root: str) -> Dict[str, List[str]]:
    eval_lists = {spk: collect_eval_files_for_speaker(data_root, spk) for spk in CLOSEDSET_SPKS}
    for spk, items in eval_lists.items():
        print(f"{spk}: {len(items)} eval utts")
    return eval_lists


# ============================================================
# 5. transcript / CER
# ============================================================
def load_ref_map(text_path: str) -> Dict[str, str]:
    ref_map: Dict[str, str] = {}
    with open(text_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if not line:
            continue

        m = re.match(r'\(\s*(arctic_[ab]\d+)\s+"(.+)"\s*\)', line)
        if m:
            utt_id = m.group(1)
            text = m.group(2)
            ref_map[utt_id] = normalize_text(text)
            continue

        if "|" in line:
            parts = line.split("|", 1)
            utt_id = parts[0].strip()
            text = parts[1].strip()
            ref_map[utt_id] = normalize_text(text)
            continue

        if "\t" in line:
            parts = line.split("\t", 1)
            utt_id = parts[0].strip()
            text = parts[1].strip()
            ref_map[utt_id] = normalize_text(text)
            continue

    if len(ref_map) == 0:
        raise RuntimeError("Failed to parse transcript file. 请检查 TEXT_PATH 和文件格式。")
    return ref_map


def load_asr_model(device: str):
    from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

    print(f"[ASR] loading {ASR_MODEL_NAME}")
    processor = Wav2Vec2Processor.from_pretrained(ASR_MODEL_NAME)
    model = Wav2Vec2ForCTC.from_pretrained(ASR_MODEL_NAME).to(device)
    model.eval()
    return processor, model


@torch.no_grad()
def transcribe_wav_ctc(wav_path: str, processor, asr_model, device: str, target_sr: int = 16000) -> str:
    wav, sr = sf.read(wav_path)
    if wav.ndim == 2:
        wav = wav.mean(axis=1)
    if sr != target_sr:
        wav = librosa.resample(wav.astype(np.float32), orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    inputs = processor(wav, sampling_rate=sr, return_tensors="pt", padding=True)
    input_values = inputs.input_values.to(device)
    attention_mask = inputs.attention_mask.to(device) if "attention_mask" in inputs else None

    logits = asr_model(input_values, attention_mask=attention_mask).logits
    pred_ids = torch.argmax(logits, dim=-1)
    text = processor.batch_decode(pred_ids)[0]
    return normalize_text(text)


def evaluate_cer(meta_df: pd.DataFrame, ref_map: Dict[str, str], processor, asr_model, device: str, out_dir: str) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, float]]:
    from jiwer import cer

    rows = []
    for item in tqdm(meta_df.to_dict("records"), desc="CER"):
        utt_id = item["utt_id"]
        wav_path = item["wav_out_path"]

        if utt_id not in ref_map:
            print("[CER skip no ref]", utt_id)
            continue

        ref = ref_map[utt_id]
        hyp = transcribe_wav_ctc(wav_path, processor, asr_model, device)
        cur_cer = cer(ref, hyp)

        rows.append({
            "src_spk": item["src_spk"],
            "tgt_spk": item["tgt_spk"],
            "utt_id": utt_id,
            "ref": ref,
            "hyp": hyp,
            "cer": float(cur_cer),
            "wav_path": wav_path,
        })

    cer_df = pd.DataFrame(rows)
    detail_csv = os.path.join(out_dir, "cer_detail.csv")
    cer_df.to_csv(detail_csv, index=False, encoding="utf-8-sig")

    summary_rows = []
    for src_spk, tgt_spk in PAIR_ORDER:
        sub = cer_df[(cer_df["src_spk"] == src_spk) & (cer_df["tgt_spk"] == tgt_spk)]
        summary_rows.append({
            "src_spk": src_spk,
            "tgt_spk": tgt_spk,
            "count": int(len(sub)),
            "cer_mean_pct": float(sub["cer"].mean() * 100.0) if len(sub) > 0 else np.nan,
            "cer_std_pct": float(sub["cer"].std(ddof=0) * 100.0) if len(sub) > 0 else np.nan,
        })

    cer_summary = pd.DataFrame(summary_rows)
    cer_summary.to_csv(os.path.join(out_dir, "cer_summary_by_pair.csv"), index=False, encoding="utf-8-sig")

    all_pairs = {
        "count": int(len(cer_df)),
        "cer_mean_pct": float(cer_df["cer"].mean() * 100.0) if len(cer_df) > 0 else np.nan,
        "cer_std_pct": float(cer_df["cer"].std(ddof=0) * 100.0) if len(cer_df) > 0 else np.nan,
        "cer_median_pct": float(cer_df["cer"].median() * 100.0) if len(cer_df) > 0 else np.nan,
    }

    with open(os.path.join(out_dir, "cer_summary_final.txt"), "w", encoding="utf-8") as f:
        f.write("CER [%] COMPARISONS (Your VoiceGrad checkpoint)\n\n")
        for src_spk in ["bdl", "clb", "slt", "rms"]:
            sub = cer_summary[cer_summary["src_spk"] == src_spk]
            f.write(f"{src_spk}\n")
            for _, row in sub.iterrows():
                f.write(
                    f"  {row['tgt_spk']}: {row['cer_mean_pct']:.2f} ± {row['cer_std_pct']:.2f} (n={int(row['count'])})\n"
                )
            f.write("\n")
        f.write(
            f"All pairs: {all_pairs['cer_mean_pct']:.2f} ± {all_pairs['cer_std_pct']:.2f} (n={all_pairs['count']})\n"
        )

    return cer_df, cer_summary, all_pairs


# ============================================================
# 6. MCD / LFC
# ============================================================
def load_wav_mono_16k(wav_path: str, sr: int = SR) -> np.ndarray:
    wav, file_sr = sf.read(wav_path)
    if wav.ndim == 2:
        wav = wav.mean(axis=1)
    wav = wav.astype(np.float64)
    if file_sr != sr:
        wav = librosa.resample(wav.astype(np.float32), orig_sr=file_sr, target_sr=sr).astype(np.float64)
    return wav


def extract_f0_mcep(wav_path: str):
    import pyworld as pw
    import pysptk

    wav = load_wav_mono_16k(wav_path)
    f0, timeaxis = pw.harvest(
        wav,
        SR,
        frame_period=FRAME_PERIOD_MS,
        f0_floor=20.0,
        f0_ceil=600.0,
    )
    sp = pw.cheaptrick(wav, f0, timeaxis, SR)
    mcep = pysptk.sp2mc(sp, order=MCEP_ORDER, alpha=MCEP_ALPHA)
    return {
        "f0": f0.astype(np.float64),
        "mcep": mcep.astype(np.float64),
    }


def score_one_mcd_lfc(gen_wav_path: str, ref_wav_path: str) -> Tuple[float, float]:
    from scipy.spatial.distance import cdist
    from librosa.sequence import dtw

    g = extract_f0_mcep(gen_wav_path)
    r = extract_f0_mcep(ref_wav_path)

    C = cdist(g["mcep"][:, 1:], r["mcep"][:, 1:], metric="euclidean")
    _, wp = dtw(C=C)
    wp = np.array(wp[::-1], dtype=np.int64)

    dists = []
    xs, ys = [], []

    for i, j in wp:
        diff = g["mcep"][i, 1:] - r["mcep"][j, 1:]
        dists.append(MCD_CONST * np.sqrt(np.sum(diff * diff)))

        if g["f0"][i] > 0 and r["f0"][j] > 0:
            xs.append(np.log(g["f0"][i]))
            ys.append(np.log(r["f0"][j]))

    mcd = float(np.mean(dists))

    xs_arr = np.asarray(xs)
    ys_arr = np.asarray(ys)
    if len(xs_arr) < 2 or np.std(xs_arr) < 1e-12 or np.std(ys_arr) < 1e-12:
        lfc = np.nan
    else:
        lfc = float(np.corrcoef(xs_arr, ys_arr)[0, 1])

    return mcd, lfc


def build_ref_index(root: str) -> Dict[Tuple[str, str], str]:
    idx: Dict[Tuple[str, str], str] = {}
    for spk_dir in sorted(glob.glob(os.path.join(root, "*"))):
        spk = os.path.basename(spk_dir)
        if not os.path.isdir(spk_dir):
            continue

        files = glob.glob(os.path.join(spk_dir, "wav", "*.wav"))
        files += glob.glob(os.path.join(spk_dir, "*.wav"))
        if not files:
            files = glob.glob(os.path.join(spk_dir, "**", "*.wav"), recursive=True)

        for wav_path in files:
            utt_id = os.path.splitext(os.path.basename(wav_path))[0]
            idx[(spk, utt_id)] = wav_path
    return idx


def mean_ci95(arr) -> Tuple[float, float, int]:
    arr = pd.Series(arr).dropna().to_numpy(dtype=np.float64)
    n = len(arr)
    if n == 0:
        return np.nan, np.nan, 0
    mean = float(np.mean(arr))
    std = float(np.std(arr, ddof=0))
    ci95 = float(1.96 * std / np.sqrt(n))
    return mean, ci95, n


def evaluate_mcd_lfc(meta_df: pd.DataFrame, ref_wav_root: str, out_dir: str) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, float]]:
    ref_index = build_ref_index(ref_wav_root)

    eval_df = meta_df.copy()
    eval_df["ref_wav_path"] = [
        ref_index.get((row["tgt_spk"], row["utt_id"]), None)
        for _, row in eval_df.iterrows()
    ]
    valid_df = eval_df.dropna(subset=["ref_wav_path"]).reset_index(drop=True)

    rows = []
    for _, row in tqdm(valid_df.iterrows(), total=len(valid_df), desc="MCD/LFC"):
        try:
            mcd, lfc = score_one_mcd_lfc(row["wav_out_path"], row["ref_wav_path"])
        except Exception as e:
            print("[MCD/LFC error]", row["wav_out_path"], "=>", e)
            mcd, lfc = np.nan, np.nan

        rows.append({
            "src_spk": row["src_spk"],
            "tgt_spk": row["tgt_spk"],
            "utt_id": row["utt_id"],
            "gen_wav_path": row["wav_out_path"],
            "ref_wav_path": row["ref_wav_path"],
            "mcd_db": mcd,
            "lfc": lfc,
        })

    detail_df = pd.DataFrame(rows)
    detail_df.to_csv(os.path.join(out_dir, "mcd_lfc_detail.csv"), index=False, encoding="utf-8-sig")

    pair_rows = []
    for src_spk, tgt_spk in PAIR_ORDER:
        sub = detail_df[(detail_df["src_spk"] == src_spk) & (detail_df["tgt_spk"] == tgt_spk)]
        mcd_mean, mcd_ci95, mcd_n = mean_ci95(sub["mcd_db"])
        lfc_mean, lfc_ci95, lfc_n = mean_ci95(sub["lfc"])
        pair_rows.append({
            "src_spk": src_spk,
            "tgt_spk": tgt_spk,
            "count": int(len(sub)),
            "mcd_mean_db": mcd_mean,
            "mcd_ci95_db": mcd_ci95,
            "lfc_mean": lfc_mean,
            "lfc_ci95": lfc_ci95,
            "valid_mcd_n": mcd_n,
            "valid_lfc_n": lfc_n,
        })

    pair_summary = pd.DataFrame(pair_rows)
    pair_summary.to_csv(os.path.join(out_dir, "mcd_lfc_summary_by_pair.csv"), index=False, encoding="utf-8-sig")

    all_mcd_mean, all_mcd_ci95, all_mcd_n = mean_ci95(detail_df["mcd_db"])
    all_lfc_mean, all_lfc_ci95, all_lfc_n = mean_ci95(detail_df["lfc"])

    all_pairs = {
        "all_pairs_mcd_db": all_mcd_mean,
        "all_pairs_mcd_ci95_db": all_mcd_ci95,
        "all_pairs_mcd_n": all_mcd_n,
        "all_pairs_lfc": all_lfc_mean,
        "all_pairs_lfc_ci95": all_lfc_ci95,
        "all_pairs_lfc_n": all_lfc_n,
    }
    return detail_df, pair_summary, all_pairs


# ============================================================
# 7. pMOS（沿用你之前 notebook 的 dnsmos）
# ============================================================
def load_dnsmos_if_needed():
    if not USE_DNSMOS_PMOS:
        return None
    from speechmos import dnsmos
    return dnsmos


def evaluate_pmos(meta_df: pd.DataFrame, dnsmos_module, out_dir: str) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, float]]:
    rows = []
    for _, row in tqdm(meta_df.iterrows(), total=len(meta_df), desc="pMOS"):
        wav_path = row["wav_out_path"]
        try:
            audio, sr = librosa.load(wav_path, sr=16000)
            scores = dnsmos_module.run(audio, sr)
            pmos_score = float(scores["ovrl_mos"])
        except Exception as e:
            print("[DNSMOS error]", wav_path, "=>", e)
            pmos_score = np.nan

        rows.append({
            "src_spk": row["src_spk"],
            "tgt_spk": row["tgt_spk"],
            "utt_id": row["utt_id"],
            "wav_path": wav_path,
            "pmos": pmos_score,
        })

    detail_df = pd.DataFrame(rows)
    detail_df.to_csv(os.path.join(out_dir, "pmos_dnsmos_detail.csv"), index=False, encoding="utf-8-sig")

    pair_rows = []
    for src_spk, tgt_spk in PAIR_ORDER:
        sub = detail_df[(detail_df["src_spk"] == src_spk) & (detail_df["tgt_spk"] == tgt_spk)]
        mean_val, ci95_val, n_val = mean_ci95(sub["pmos"])
        pair_rows.append({
            "src_spk": src_spk,
            "tgt_spk": tgt_spk,
            "count": int(len(sub)),
            "pmos_mean": mean_val,
            "pmos_ci95": ci95_val,
            "valid_n": n_val,
        })

    pair_summary = pd.DataFrame(pair_rows)
    pair_summary.to_csv(os.path.join(out_dir, "pmos_summary_by_pair.csv"), index=False, encoding="utf-8-sig")

    pmos_mean, pmos_ci95, pmos_n = mean_ci95(detail_df["pmos"])
    all_pairs = {
        "all_pairs_pmos": pmos_mean,
        "all_pairs_pmos_ci95": pmos_ci95,
        "all_pairs_pmos_n": pmos_n,
        "all_pairs_pmos_std": float(detail_df["pmos"].std(ddof=0)) if len(detail_df) > 0 else np.nan,
    }
    return detail_df, pair_summary, all_pairs


# ============================================================
# 8. checkpoint 路径解析
# ============================================================
def resolve_ckpt_path(ckpt_item, ckpt_dir: str) -> str:
    if isinstance(ckpt_item, int):
        candidates = [
            os.path.join(ckpt_dir, f"model_epoch_{ckpt_item}.pt"),
            os.path.join(ckpt_dir, f"epoch_{ckpt_item}.pt"),
            os.path.join(ckpt_dir, f"ckpt_{ckpt_item}.pt"),
            os.path.join(ckpt_dir, f"checkpoint_{ckpt_item}.pt"),
        ]
        for p in candidates:
            if os.path.exists(p):
                return p
        raise FileNotFoundError(f"找不到 epoch={ckpt_item} 的 checkpoint。尝试过: {candidates}")

    if isinstance(ckpt_item, str) and ckpt_item.lower() == "best":
        candidates = [
            os.path.join(ckpt_dir, "best.pt"),
            os.path.join(ckpt_dir, "model_best.pt"),
            os.path.join(ckpt_dir, "best_ckpt.pt"),
            os.path.join(ckpt_dir, "checkpoint_best.pt"),
        ]
        for p in candidates:
            if os.path.exists(p):
                return p

        globbed = sorted(glob.glob(os.path.join(ckpt_dir, "*best*.pt")))
        if globbed:
            return globbed[0]
        raise FileNotFoundError(f"找不到 best checkpoint。尝试过: {candidates}")

    if isinstance(ckpt_item, str) and os.path.exists(ckpt_item):
        return ckpt_item

    raise ValueError(f"无法解析 checkpoint: {ckpt_item}")


def ckpt_tag_from_item(ckpt_item) -> str:
    return f"ckpt_{ckpt_item}" if isinstance(ckpt_item, int) else str(ckpt_item)


# ============================================================
# 9. 模型 / vocoder
# ============================================================
def build_model_and_diffusion(VoiceGrad, VoiceGradDiffusion, device: str = DEVICE):
    model = VoiceGrad(
        n_mels=80,
        n_bnf=144,
        n_channels=512,
        n_spk=4,
    ).to(device)

    diffusion = VoiceGradDiffusion(n_levels=20, offset=0.008).to(device)
    return model, diffusion


def load_ckpt_to_model(ckpt_path: str, model, map_location: str = "cpu"):
    ckpt = torch.load(ckpt_path, map_location=map_location)

    if isinstance(ckpt, dict) and "model" in ckpt:
        state_dict = ckpt["model"]
        epoch = ckpt.get("epoch", "unknown")
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
        epoch = ckpt.get("epoch", "unknown")
    else:
        state_dict = ckpt
        epoch = "unknown"

    model.load_state_dict(state_dict, strict=True)
    return epoch, ckpt


def load_hifigan(device: str = DEVICE):
    if HIFIGAN_ROOT not in sys.path:
        sys.path.append(HIFIGAN_ROOT)

    from env import AttrDict  # type: ignore
    from models import Generator  # type: ignore

    with open(HIFIGAN_CONFIG, "r", encoding="utf-8") as f:
        h = AttrDict(json.load(f))

    generator = Generator(h).to(device)
    ckpt = torch.load(HIFIGAN_CKPT, map_location=device)

    if isinstance(ckpt, dict) and "generator" in ckpt:
        generator.load_state_dict(ckpt["generator"])
    else:
        generator.load_state_dict(ckpt)

    generator.eval()
    generator.remove_weight_norm()
    return generator, h


@torch.no_grad()
def convert_one(model, diffusion, src_spk: str, src_mel_file: str, tgt_spk: str, mel_mean: torch.Tensor, mel_std: torch.Tensor):
    mel_np, bnf_np, _, _ = load_sample_from_data(DATA_ROOT, src_spk, src_mel_file)
    mel_norm = normalize_mel(mel_np, mel_mean, mel_std).to(DEVICE)
    bnf_tensor = torch.from_numpy(bnf_np).float().unsqueeze(0).to(DEVICE)
    target_id = torch.tensor([SPK2ID[tgt_spk]], dtype=torch.long, device=DEVICE)

    converted_norm = diffusion.sample(
        model=model,
        x_source=mel_norm,
        speaker_id=target_id,
        bnf=bnf_tensor,
        start_level=START_LEVEL,
    )

    converted_mel = denormalize_mel(converted_norm.cpu(), mel_mean, mel_std)
    return converted_mel


def generate_closedset_wavs_for_ckpt(
    ckpt_path: str,
    ckpt_tag: str,
    VoiceGrad,
    VoiceGradDiffusion,
    generator,
    h,
    eval_lists: Dict[str, List[str]],
    mel_mean: torch.Tensor,
    mel_std: torch.Tensor,
) -> Tuple[pd.DataFrame, str, str]:
    ckpt_root = os.path.join(OUT_BASE, ckpt_tag)
    wav_root = os.path.join(ckpt_root, "generated_wavs")
    metric_root = os.path.join(ckpt_root, "metrics")
    os.makedirs(wav_root, exist_ok=True)
    os.makedirs(metric_root, exist_ok=True)

    meta_csv = os.path.join(ckpt_root, "generated_meta.csv")
    done_flag = os.path.join(ckpt_root, "GEN_DONE.ok")

    if REUSE_EXISTING_WAVS and os.path.exists(meta_csv) and os.path.exists(done_flag):
        meta_df = pd.read_csv(meta_csv)
        if len(meta_df) == len(PAIR_ORDER) * 32:
            print(f"[{ckpt_tag}] 已存在完整生成结果，直接复用 wav。")
            return meta_df, ckpt_root, metric_root

    print(f"\n[Generate] {ckpt_tag}")
    model, diffusion = build_model_and_diffusion(VoiceGrad, VoiceGradDiffusion, device=DEVICE)
    epoch, _ = load_ckpt_to_model(ckpt_path, model, map_location=DEVICE)
    model.eval()
    print(f"Loaded ckpt: {ckpt_path}")
    print(f"Checkpoint epoch field: {epoch}")

    all_meta = []
    expected = len(PAIR_ORDER) * 32

    for src_spk, tgt_spk in PAIR_ORDER:
        pair_dir = os.path.join(wav_root, f"{src_spk}_to_{tgt_spk}")
        os.makedirs(pair_dir, exist_ok=True)
        src_files = eval_lists[src_spk]
        print(f"Running pair: {src_spk} -> {tgt_spk} | n={len(src_files)}")

        for src_mel_file in tqdm(src_files, leave=False, desc=f"{src_spk}->{tgt_spk}"):
            base = src_mel_file.replace(".npy", "")
            wav_out_path = os.path.join(pair_dir, f"{base}.wav")

            if REUSE_EXISTING_WAVS and os.path.exists(wav_out_path):
                all_meta.append({
                    "src_spk": src_spk,
                    "tgt_spk": tgt_spk,
                    "utt_id": base,
                    "wav_out_path": wav_out_path,
                })
                continue

            converted_mel = convert_one(model, diffusion, src_spk, src_mel_file, tgt_spk, mel_mean, mel_std)
            y_g_hat = generator(converted_mel.to(DEVICE))
            wav = y_g_hat.squeeze().detach().cpu().numpy()
            sf.write(wav_out_path, wav, h.sampling_rate)

            all_meta.append({
                "src_spk": src_spk,
                "tgt_spk": tgt_spk,
                "utt_id": base,
                "wav_out_path": wav_out_path,
            })

    meta_df = pd.DataFrame(all_meta)
    meta_df.to_csv(meta_csv, index=False, encoding="utf-8-sig")

    if len(meta_df) != expected:
        print(f"[Warning] {ckpt_tag}: expected {expected}, got {len(meta_df)}")
    else:
        Path(done_flag).write_text("ok\n", encoding="utf-8")
        print(f"[{ckpt_tag}] generation finished: {len(meta_df)} wavs")

    del model, diffusion
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return meta_df, ckpt_root, metric_root


# ============================================================
# 10. 单个 checkpoint 的完整评测
# ============================================================
def evaluate_one_ckpt(
    ckpt_item,
    VoiceGrad,
    VoiceGradDiffusion,
    generator,
    h,
    eval_lists,
    mel_mean,
    mel_std,
    ref_map,
    processor,
    asr_model,
    dnsmos_module,
) -> Dict[str, object]:
    ckpt_path = resolve_ckpt_path(ckpt_item, CKPT_DIR)
    ckpt_tag = ckpt_tag_from_item(ckpt_item)
    ckpt_root = os.path.join(OUT_BASE, ckpt_tag)
    metric_root = os.path.join(ckpt_root, "metrics")
    os.makedirs(metric_root, exist_ok=True)

    final_summary_csv = os.path.join(metric_root, "summary_all_metrics.csv")
    if SKIP_IF_DONE and os.path.exists(final_summary_csv):
        print(f"[{ckpt_tag}] 检测到 summary_all_metrics.csv，跳过重跑。")
        row = pd.read_csv(final_summary_csv).iloc[0].to_dict()
        return row

    meta_df, ckpt_root, metric_root = generate_closedset_wavs_for_ckpt(
        ckpt_path=ckpt_path,
        ckpt_tag=ckpt_tag,
        VoiceGrad=VoiceGrad,
        VoiceGradDiffusion=VoiceGradDiffusion,
        generator=generator,
        h=h,
        eval_lists=eval_lists,
        mel_mean=mel_mean,
        mel_std=mel_std,
    )

    cer_df, cer_pair_summary, cer_all = evaluate_cer(
        meta_df=meta_df,
        ref_map=ref_map,
        processor=processor,
        asr_model=asr_model,
        device=DEVICE,
        out_dir=metric_root,
    )

    mcd_lfc_df, mcd_lfc_pair_summary, mcd_lfc_all = evaluate_mcd_lfc(
        meta_df=meta_df,
        ref_wav_root=os.path.join(DATA_ROOT, "wav"),
        out_dir=metric_root,
    )

    if dnsmos_module is not None:
        pmos_df, pmos_pair_summary, pmos_all = evaluate_pmos(
            meta_df=meta_df,
            dnsmos_module=dnsmos_module,
            out_dir=metric_root,
        )
    else:
        pmos_df = pd.DataFrame()
        pmos_pair_summary = pd.DataFrame()
        pmos_all = {
            "all_pairs_pmos": np.nan,
            "all_pairs_pmos_ci95": np.nan,
            "all_pairs_pmos_n": 0,
            "all_pairs_pmos_std": np.nan,
        }

    one_row = {
        "ckpt_tag": ckpt_tag,
        "ckpt_item": str(ckpt_item),
        "ckpt_path": ckpt_path,
        "generated_wav_count": int(len(meta_df)),
        "cer_count": int(cer_all["count"]),
        "cer_mean_pct": cer_all["cer_mean_pct"],
        "cer_std_pct": cer_all["cer_std_pct"],
        "cer_median_pct": cer_all["cer_median_pct"],
        "mcd_mean_db": mcd_lfc_all["all_pairs_mcd_db"],
        "mcd_ci95_db": mcd_lfc_all["all_pairs_mcd_ci95_db"],
        "lfc_mean": mcd_lfc_all["all_pairs_lfc"],
        "lfc_ci95": mcd_lfc_all["all_pairs_lfc_ci95"],
        "pmos_mean": pmos_all["all_pairs_pmos"],
        "pmos_ci95": pmos_all["all_pairs_pmos_ci95"],
        "pmos_std": pmos_all["all_pairs_pmos_std"],
        "ckpt_root": ckpt_root,
        "metric_root": metric_root,
        "cer_detail_csv": os.path.join(metric_root, "cer_detail.csv"),
        "cer_pair_csv": os.path.join(metric_root, "cer_summary_by_pair.csv"),
        "mcd_lfc_detail_csv": os.path.join(metric_root, "mcd_lfc_detail.csv"),
        "mcd_lfc_pair_csv": os.path.join(metric_root, "mcd_lfc_summary_by_pair.csv"),
        "pmos_detail_csv": os.path.join(metric_root, "pmos_dnsmos_detail.csv"),
        "pmos_pair_csv": os.path.join(metric_root, "pmos_summary_by_pair.csv"),
    }

    pd.DataFrame([one_row]).to_csv(final_summary_csv, index=False, encoding="utf-8-sig")
    print(f"[{ckpt_tag}] summary saved -> {final_summary_csv}")
    return one_row


# ============================================================
# 11. 主流程
# ============================================================
def main() -> None:
    print_config()
    ensure_paths()
    prepare_project_imports(PROJECT_ROOT)

    VoiceGrad, VoiceGradDiffusion = import_project_modules()

    print("\nLoading shared resources...")
    mel_mean, mel_std = load_stats(DATA_ROOT)
    eval_lists = get_closedset_eval_lists(DATA_ROOT)
    ref_map = load_ref_map(TEXT_PATH)
    processor, asr_model = load_asr_model(DEVICE)
    dnsmos_module = load_dnsmos_if_needed() if USE_DNSMOS_PMOS else None
    generator, h = load_hifigan(DEVICE)

    results = []
    for ckpt_item in EVAL_CKPTS:
        print("\n" + "#" * 90)
        print(f"Start checkpoint: {ckpt_item}")
        print("#" * 90)
        row = evaluate_one_ckpt(
            ckpt_item=ckpt_item,
            VoiceGrad=VoiceGrad,
            VoiceGradDiffusion=VoiceGradDiffusion,
            generator=generator,
            h=h,
            eval_lists=eval_lists,
            mel_mean=mel_mean,
            mel_std=mel_std,
            ref_map=ref_map,
            processor=processor,
            asr_model=asr_model,
            dnsmos_module=dnsmos_module,
        )
        results.append(row)

    summary_df = pd.DataFrame(results)

    # 排名：MCD/CER 越小越好；LFC/pMOS 越大越好
    if len(summary_df) > 0:
        summary_df["rank_mcd"] = summary_df["mcd_mean_db"].rank(method="min", ascending=True)
        summary_df["rank_cer"] = summary_df["cer_mean_pct"].rank(method="min", ascending=True)
        summary_df["rank_lfc"] = summary_df["lfc_mean"].rank(method="min", ascending=False)
        summary_df["rank_pmos"] = summary_df["pmos_mean"].rank(method="min", ascending=False)
        summary_df["rank_avg"] = summary_df[["rank_mcd", "rank_cer", "rank_lfc", "rank_pmos"]].mean(axis=1)
        summary_df = summary_df.sort_values(["rank_avg", "mcd_mean_db", "cer_mean_pct"], ascending=[True, True, True]).reset_index(drop=True)

    summary_csv = os.path.join(OUT_BASE, "all_ckpts_summary.csv")
    summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

    summary_txt = os.path.join(OUT_BASE, "all_ckpts_summary.txt")
    with open(summary_txt, "w", encoding="utf-8") as f:
        f.write("VoiceGrad closed-set checkpoint sweep summary (exclude ckpt1000)\n\n")
        for _, row in summary_df.iterrows():
            f.write(
                f"{row['ckpt_tag']}: "
                f"CER={row['cer_mean_pct']:.3f}%, "
                f"MCD={row['mcd_mean_db']:.4f}, "
                f"LFC={row['lfc_mean']:.4f}, "
                f"pMOS={row['pmos_mean']:.4f}, "
                f"rank_avg={row['rank_avg']:.3f}\n"
            )

    print("\n" + "=" * 100)
    print("全部 checkpoint 评测完成")
    print("Summary CSV:", summary_csv)
    print("Summary TXT:", summary_txt)
    print("=" * 100)
    print(summary_df[[
        "ckpt_tag", "cer_mean_pct", "mcd_mean_db", "lfc_mean", "pmos_mean",
        "rank_mcd", "rank_cer", "rank_lfc", "rank_pmos", "rank_avg"
    ]])


if __name__ == "__main__":
    main()


PROJECT_ROOT : /content/drive/MyDrive/VoiceGrad
DATA_ROOT    : /content/drive/MyDrive/VoiceGrad/data
CKPT_DIR     : /content/drive/MyDrive/VoiceGrad/checkpoints
HIFIGAN_ROOT : /content/drive/MyDrive/VoiceGrad/hifi-gan
TEXT_PATH    : /content/drive/MyDrive/VoiceGrad/data/cmuarctic.data.txt
OUT_BASE     : /content/drive/MyDrive/VoiceGrad/ckpt_sweep_closedset_eval_except1000
DEVICE       : cuda
EVAL_CKPTS   : [500, 1500, 2000, 2500, 3000, 3500, 4000, 'best']
START_LEVEL  : 11

Loading shared resources...
clb: 32 eval utts
bdl: 32 eval utts
slt: 32 eval utts
rms: 32 eval utts
[ASR] loading facebook/wav2vec2-large-960h-lv60-self


Loading weights:   0%|          | 0/423 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-large-960h-lv60-self
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Removing weight norm...

##########################################################################################
Start checkpoint: 500
##########################################################################################

[Generate] ckpt_500
Loaded ckpt: /content/drive/MyDrive/VoiceGrad/checkpoints/model_epoch_500.pt
Checkpoint epoch field: 500
Running pair: bdl -> clb | n=32


Running pair: bdl -> slt | n=32


...

Running pair: rms -> bdl | n=32


Running pair: rms -> slt | n=32


[best] generation finished: 384 wavs


pMOS: 100%|██████████| 384/384 [04:05<00:00,  1.56it/s]

[best] summary saved -> /content/drive/MyDrive/VoiceGrad/ckpt_sweep_closedset_eval_except1000/best/metrics/summary_all_metrics.csv

全部 checkpoint 评测完成
Summary CSV: /content/drive/MyDrive/VoiceGrad/ckpt_sweep_closedset_eval_except1000/all_ckpts_summary.csv
Summary TXT: /content/drive/MyDrive/VoiceGrad/ckpt_sweep_closedset_eval_except1000/all_ckpts_summary.txt
    ckpt_tag  cer_mean_pct  mcd_mean_db  lfc_mean  pmos_mean  rank_mcd  \
0  ckpt_1500      2.208122     6.062295  0.444491   3.198459       6.0   
1  ckpt_2000      2.209479     6.047547  0.472407   3.195830       5.0   
2  ckpt_2500      2.244921     6.031280  0.431574   3.197313       4.0   
3  ckpt_3000      2.210512     6.008269  0.416071   3.194875       1.0   
4   ckpt_500      1.916329     6.583779  0.434710   3.127703       7.0   
5  ckpt_4000      2.251827     6.015563  0.428681   3.179068       2.0   
6  ckpt_3500      2.286303     6.018991  0.429504   3.190421       3.0   
7       best      2.572149     7.864434  0.3757